# Notebook 3: Read the test pages and make the Kaggle file

This reads all 208 test pages with your best run from Notebook 2 (survey spreads one page at a
time, as in training) and saves `submission.csv` in the format Kaggle wants: `page_id`, `text`,
one row per test page.

About 40 to 60 minutes on an `ml.g5.xlarge`. With `WHERE = "job"` (Step 4) the reading runs in a
SageMaker training job, so the GPU is billed only while it reads and this notebook can be a small
`ml.t3.medium`. If it stops partway, run the cell again: it carries
on where it stopped. Run the cells from top to bottom with **Shift + Enter**. Kernel: **conda_pytorch_p310** (or another `conda_pytorch` kernel, not `conda_python3`).

## Step 1. Install the libraries

The same as Notebook 2, Step 1.

In [ ]:
%pip install -q -r requirements.txt

import importlib.metadata
import importlib.util
import subprocess
import sys

if importlib.util.find_spec("torch") is None:
    raise SystemExit("This kernel has no PyTorch, so it can't run the model. Choose Kernel → Change Kernel, "
                     "pick conda_pytorch_p310 (or another conda_pytorch kernel), then run this cell again.")

# The model also needs torchvision in exactly the version and build made for this kernel's
# PyTorch (2.8.0 goes with 0.23.0, 2.9.1 with 0.24.1, ...). conda_pytorch kernels have it.
def torchvision_works():
    return subprocess.run([sys.executable, "-c", "import torchvision"], capture_output=True).returncode == 0

torch_version, _, build = importlib.metadata.version("torch").partition("+")
if not torchvision_works():
    _, minor, patch = (torch_version.split(".") + ["0"])[:3]
    wanted = f"torchvision==0.{int(minor) + 15}.{patch}"
    sources = ([["--index-url", f"https://download.pytorch.org/whl/{build}"]] if build else []) + [[]]
    for source in sources:
        print(f"Installing {wanted} for PyTorch {torch_version}{'+' + build if build else ''} ...")
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", "--no-deps", wanted, *source])
        if torchvision_works():
            break
        subprocess.run([sys.executable, "-m", "pip", "uninstall", "-q", "-y", "torchvision"])
    else:
        raise SystemExit("Couldn't install a torchvision that works with this kernel's PyTorch. Switch the "
                         "kernel to conda_pytorch_p310 (Kernel → Change Kernel) and run this cell again.")
print(f"PyTorch {importlib.metadata.version('torch')}, torchvision {importlib.metadata.version('torchvision')}: ready.")

## Step 2. Set up the folders

In [ ]:
import importlib.util
import os
import subprocess
import sys

# Which kernel is this? It should be conda_pytorch_p310 (or another conda_pytorch kernel).
print("Kernel environment:", os.path.basename(os.path.dirname(os.path.dirname(sys.executable))))
missing = [pkg for module, pkg in (("PIL", "pillow"), ("pandas", "pandas"))
           if importlib.util.find_spec(module) is None]
if missing:
    print("Installing", ", ".join(missing), "into this kernel ...")
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", *missing], check=True)

# Everything this project downloads or creates lives here, on the notebook
# instance's storage volume, so it survives Stop and Start.
WORK = "/home/ec2-user/SageMaker/churro-work"
DATA = os.path.join(WORK, "data")     # page images and lists of pages
RUNS = os.path.join(WORK, "runs")     # one folder per training run
SCRIPTS = os.path.abspath("scripts")  # the scripts folder next to this notebook
for folder in (WORK, DATA, RUNS):
    os.makedirs(folder, exist_ok=True)
sys.path.insert(0, SCRIPTS)
os.environ["HF_HOME"] = os.path.join(WORK, "hf_cache")   # the 7.5 GB model download goes here

# Your bucket name is saved here, outside the repository, so it's typed once and
# updating the code (git pull) never touches it.
import json
SETTINGS = os.path.join(WORK, "my_settings.json")
def saved_setting(name, default=""):
    try:
        return json.load(open(SETTINGS)).get(name, default)
    except (OSError, ValueError):
        return default
def save_setting(name, value):
    try:
        current = json.load(open(SETTINGS))
    except (OSError, ValueError):
        current = {}
    current[name] = value
    json.dump(current, open(SETTINGS, "w"), indent=2)
print("Working folder:", WORK)

## Step 3. Pick a run for each collection

Your runs, with each collection's dev CER. Each collection can be read by a different run: for
example a `"kade-final"` run for the Kade pages, a `"survey-final"` run for the survey pages, and a
`"full"` run for Dominy. By default the run with the best mean is used for everything; type run
names into `USE` to choose. Smoke runs aren't listed. The test pages are read exactly the way each
chosen run read its dev pages, with the same settings.

Runs trained with Kade documents held out (`kadeA-...`, `kadeB-...`) are for choosing settings.
For the test pages, use a `"kade-final"` run trained on every Kade page.


In [ ]:
import json
import pandas as pd

runs = []
for name in os.listdir(RUNS):
    path = os.path.join(RUNS, name, "summary.json")
    if os.path.exists(path) and name != "smoke":
        s = json.load(open(path))
        runs.append({"run": name, **s["dev_cer"], "dev pages": s["dev_pages"],
                     "trained on": s.get("categories", "all"), "held out": s.get("holdout_docs", "")})
if not runs:
    raise SystemExit("No finished runs yet. Run Notebook 2 first.")
runs = pd.DataFrame(runs).sort_values("mean of collections")
print(runs.round(3).to_string(index=False))

best = runs[runs["trained on"] == "all"].iloc[0]["run"] if (runs["trained on"] == "all").any() else runs.iloc[0]["run"]
USE = {                      # <-- the run for each collection's test pages
    "kade_letters": best,
    "survey_notes": best,
    "dominy_accounts": best,
}
for collection, run in USE.items():
    s = json.load(open(os.path.join(RUNS, run, "summary.json")))
    if s.get("categories", "all") != "all" and collection not in s["categories"].split(","):
        print(f"WARNING: {run} was trained on {s['categories']}, not {collection}.")
    if s.get("holdout_docs"):
        print(f"Note: {run} was trained with {s['holdout_docs']} held out. A *-final run is better for the test pages.")
print("\nUsing:", json.dumps(USE, indent=2))


## Step 4. Read the test pages

Each chosen run reads its collections' pages. With `WHERE = "job"` each run's reading is one
training job (5 to 10 minutes of start-up each). The job keeps going if the browser disconnects,
and running the cell again carries on where it stopped. The last number on each line is the time left.


In [ ]:
import hashlib

WHERE = "job"                   # "job": a SageMaker training job reads the pages; "here": this instance's GPU
JOB_INSTANCE = "ml.g5.xlarge"
JOB_HOURS = 3                   # cost cap per job

label = next(iter(USE.values())) if len(set(USE.values())) == 1 else \
    "combined-" + hashlib.md5(json.dumps(USE, sort_keys=True).encode()).hexdigest()[:8]
out_dir = os.path.join(WORK, "predictions", label)
os.makedirs(out_dir, exist_ok=True)
json.dump(USE, open(os.path.join(out_dir, "runs_used.json"), "w"), indent=2)
python = sys.executable
raws = []
import glob
tests_all = pd.read_csv(f"{DATA}/test_pages.csv", keep_default_na=False)
for run in sorted(set(USE.values())):
    collections = ",".join(c for c, r in USE.items() if r == run)
    raw = os.path.join(out_dir, f"test_raw_{run}.csv")
    raws.append(raw)
    print(f"=== {run}: {collections}")
    # This run's reading of these pages in an earlier Notebook 3 folder is reused instead of read (and paid for) again
    wanted = set(tests_all.page_id[tests_all.category.isin(collections.split(","))])
    for earlier in glob.glob(os.path.join(WORK, "predictions", "*", f"test_raw_{run}.csv")):
        e = pd.read_csv(earlier, keep_default_na=False)
        if not os.path.exists(raw) and wanted <= set(e.page_id):
            e[e.page_id.isin(wanted)].to_csv(raw, index=False)
            print("Reusing its reading in", os.path.relpath(earlier, WORK))
    if os.path.exists(raw) and wanted <= set(pd.read_csv(raw, keep_default_na=False).page_id):
        continue
    if WHERE == "job":
        import sagemaker_job
        sagemaker_job.run_job("transcribe.py", {"data_dir": DATA, "pages": f"{DATA}/test_pages.csv", "out": raw,
                                                "run_dir": os.path.join(RUNS, run), "categories": collections},
                              out_dir, work=WORK, bucket=saved_setting("bucket"), run_name=f"test-{run}",
                              instance_type=JOB_INSTANCE, max_hours=JOB_HOURS, log_name=f"transcribe_{run}.log")
    else:
        !{python} {SCRIPTS}/transcribe.py --data_dir {DATA} --pages {DATA}/test_pages.csv --out {raw} --run_dir {os.path.join(RUNS, run)} --categories {collections}


## Step 5. Make `submission.csv`

The rows follow `sample_submission.csv` (or `test.csv` if there isn't one), with exactly the two
columns Kaggle accepts. It checks that every test page has text.

In [ ]:
preds = pd.concat([pd.read_csv(r, keep_default_na=False) for r in raws], ignore_index=True)
sample = os.path.join(DATA, "sample_submission.csv")
order = pd.read_csv(sample if os.path.exists(sample) else os.path.join(DATA, "test.csv"), keep_default_na=False)[["page_id"]]
submission = order.merge(preds[["page_id", "text"]], on="page_id", how="left").fillna({"text": ""})
empty = submission[submission.text.str.strip() == ""]
if len(empty):
    print(f"WARNING: {len(empty)} pages have no text, e.g. {', '.join(empty.page_id[:5])}. Run Step 4 again.")
path = os.path.join(out_dir, "submission.csv")
submission.to_csv(path, index=False)
print(f"{len(submission)} pages saved to {path}")
print(submission.assign(collection=submission.page_id.str.split("_").str[0]).collection.value_counts().to_string())
submission.head()

## Step 5b (optional). Cross-check Dominy with a second model

Two models read every Dominy test page, then **each model judges both readings**: it is shown the
page and a candidate text, and says how likely that text is (the mean log-probability of its
tokens, without writing anything). The reading with the better mean judgement of the two models
is kept. A model's own sureness was a poor judge (it liked its own runaway and degenerate
readings), but a second model doesn't share the first one's habits. A reading stuck repeating
itself is never kept while the other isn't. On 19 Nov the best possible pick between CHURRO and
olmOCR readings of the held-out Dominy pages was 0.217 against 0.303 for the best single model, so
there is room; this step measures how much of it the judging finds.

`CROSS` names the models, the current best first (it keeps a page unless the other reading is
judged better by more than `CROSS_MARGIN`). Both runs should hold out the same Dominy documents
(`DOMINY_HOLDOUT` in Notebook 2): their held-out pages are judged too, and `cross_report.json`
shows the CER of each model alone, of the cross-check, and of the best possible pick on them.
**Use the cross-checked file only if the cross-check beats "only churro" there.**

A run with no test readings yet reads the Dominy test pages first (one job), then a second job
judges (about 15 to 25 minutes, mostly loading the two models). The result is
`predictions/<this label>/cross-<...>/submission.csv`: Step 5's file with the Dominy rows replaced.


In [ ]:
CROSS = {                      # name: run folder in churro-work/runs. The first one wins a tie.
    "churro": USE["dominy_accounts"],
    "olmocr": "",              # an olmOCR run (Notebook 2 with MODEL = "olmocr-2-7b", RUN = "dominy"), "olmocr27b-dominy-..."
}
CROSS_MARGIN = 0.0             # the other reading must be judged better by more than this to be kept

import glob
import hashlib

if not all(CROSS.values()):
    raise SystemExit("Type a run name for every model in CROSS.")
for name, run in CROSS.items():
    if not os.path.exists(os.path.join(RUNS, run, "summary.json")):
        raise SystemExit(f"{name}: no finished run called {run!r} in {RUNS}.")
cross_dir = os.path.join(out_dir, "cross-" + hashlib.md5(json.dumps(CROSS, sort_keys=True).encode()).hexdigest()[:8])
os.makedirs(cross_dir, exist_ok=True)
json.dump(CROSS, open(os.path.join(cross_dir, "cross_runs.json"), "w"), indent=2)

tests = pd.read_csv(os.path.join(DATA, "test_pages.csv"), keep_default_na=False)
dominy_test = set(tests.page_id[tests.page_id.str.startswith("dominy_")])

def has_all(csv_path):
    return os.path.exists(csv_path) and dominy_test <= set(pd.read_csv(csv_path, keep_default_na=False).page_id)

held_out = {}
for name, run in CROSS.items():
    # 1. this model's Dominy test readings: from an earlier Step 4 if there is one, else read them now
    raw = os.path.join(cross_dir, f"test_raw_{run}.csv")
    if not has_all(raw):
        earlier = [f for f in glob.glob(os.path.join(WORK, "predictions", "*", f"test_raw_{run}.csv")) if has_all(f)]
        if earlier:
            print(f"{name}: using the test readings in {os.path.relpath(earlier[0], WORK)}")
            raw = earlier[0]
        else:
            print(f"=== {name}: reading the {len(dominy_test)} Dominy test pages with {run}")
            if WHERE == "job":
                import sagemaker_job
                sagemaker_job.run_job("transcribe.py", {"data_dir": DATA, "pages": f"{DATA}/test_pages.csv", "out": raw,
                                                        "run_dir": os.path.join(RUNS, run), "categories": "dominy_accounts"},
                                      cross_dir, work=WORK, bucket=saved_setting("bucket"), run_name=f"test-{run}",
                                      instance_type=JOB_INSTANCE, max_hours=JOB_HOURS, log_name=f"transcribe_{run}.log")
            else:
                !{python} {SCRIPTS}/transcribe.py --data_dir {DATA} --pages {DATA}/test_pages.csv --out {raw} --run_dir {os.path.join(RUNS, run)} --categories dominy_accounts
            if not has_all(raw):
                raise SystemExit(f"{name}: the reading of the test pages didn't finish. Run this cell again.")
    test_rows = pd.read_csv(raw, keep_default_na=False)
    test_rows = test_rows[test_rows.page_id.isin(dominy_test)][["page_id", "text"]]
    # 2. its readings of its own held-out Dominy pages, to measure the cross-check on
    dev_path = os.path.join(RUNS, run, "dev_scores.csv")
    dev_rows = pd.DataFrame(columns=["page_id", "text"])
    if os.path.exists(dev_path):
        d = pd.read_csv(dev_path, keep_default_na=False)
        dev_rows = d[d.category == "dominy_accounts"][["page_id", "prediction"]].rename(columns={"prediction": "text"})
    held_out[name] = set(dev_rows.page_id)
    pd.concat([dev_rows, test_rows], ignore_index=True).to_csv(os.path.join(cross_dir, f"candidates_{name}.csv"), index=False)
    print(f"{name}: {len(test_rows)} test pages, {len(dev_rows)} held-out pages")
common_dev = set.intersection(*held_out.values())
if any(h != common_dev for h in held_out.values()):
    print(f"NOTE: the runs held out different Dominy pages; the report uses the {len(common_dev)} they share "
          "(pages one model trained on would flatter its judgement).")
    for name in CROSS:
        c = pd.read_csv(os.path.join(cross_dir, f"candidates_{name}.csv"), keep_default_na=False)
        c[c.page_id.isin(dominy_test | common_dev)].to_csv(os.path.join(cross_dir, f"candidates_{name}.csv"), index=False)

# 3. every model judges every reading
cross_args = {"data_dir": DATA, "out_dir": cross_dir, "names": ",".join(CROSS), "margin": CROSS_MARGIN,
              **{f"run{i + 1}": os.path.join(RUNS, r) for i, r in enumerate(CROSS.values())}}
if WHERE == "job":
    import sagemaker_job
    sagemaker_job.run_job("cross_check.py", cross_args, cross_dir, work=WORK, bucket=saved_setting("bucket"),
                          run_name="cross-" + "-".join(CROSS), instance_type=JOB_INSTANCE, max_hours=JOB_HOURS,
                          log_name="cross_check.log")
else:
    flags = " ".join(f"--{k} {v}" for k, v in cross_args.items())
    !{python} {SCRIPTS}/cross_check.py {flags}

report_path = os.path.join(cross_dir, "cross_report.json")
if os.path.exists(report_path):
    rep = json.load(open(report_path))
    print(json.dumps({k: v for k, v in rep.items() if k != "by page"}, indent=2))
    print(pd.DataFrame(rep["by page"]).T.to_string())
picked = pd.read_csv(os.path.join(cross_dir, "picked.csv"), keep_default_na=False)
picked = picked[picked.page_id.isin(dominy_test)]
print("Dominy test pages kept from:", picked.picked.value_counts().to_dict())
crossed = submission.copy()
keep = dict(zip(picked.page_id, picked.text))
crossed["text"] = [keep.get(pid, t) for pid, t in zip(crossed.page_id, crossed.text)]
crossed.to_csv(os.path.join(cross_dir, "submission.csv"), index=False)
print(f"{len(crossed)} pages saved to {os.path.join(cross_dir, 'submission.csv')}")


## Step 6. Download it, or save it to S3

- **Download:** in the file browser on the left, open `churro-work/predictions/<run>/`,
  right-click `submission.csv`, choose **Download**. Then upload it on the competition's
  **Submit Prediction** page.
- **Share with the team:** run the cell below.

Then **stop the instance** if you're done.

In [ ]:
MY_BUCKET = saved_setting("bucket")   # the bucket from Notebook 1; or type another, e.g. "my-bucket"

!aws s3 sync {out_dir} s3://{MY_BUCKET}/churro-work/predictions/{label} --quiet
print(f"Saved to s3://{MY_BUCKET}/churro-work/predictions/{label}")